In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "38ebe151",
   "metadata": {},
   "source": [
    "# Paper Trading Simulation\n",
    "\n",
    "This notebook demonstrates running the StockTrader paper trading simulator\n",
    "and visualising the equity curve and key performance metrics.\n",
    "\n",
    "**Prerequisites:**\n",
    "- Sample OHLCV data in `storage/raw/` (run `python -m scripts.sample_data.download_sample`)\n",
    "- A trained model in `models/artifacts/` (run `python -m backend.prediction_engine.training.trainer`)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "16397ce3",
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys, pathlib\n",
    "\n",
    "# Ensure project root is on the path\n",
    "ROOT = pathlib.Path.cwd().parent\n",
    "if str(ROOT) not in sys.path:\n",
    "    sys.path.insert(0, str(ROOT))\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "from backend.prediction_engine.feature_store.feature_store import build_features\n",
    "from backend.prediction_engine.models.lightgbm_model import LightGBMModel\n",
    "from backend.trading_engine.simulator import PaperSimulator, OrderIntent"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f5b8e808",
   "metadata": {},
   "source": [
    "## 1. Load Model and Sample Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "987f5b4c",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the latest trained model\n",
    "import json\n",
    "\n",
    "registry_path = ROOT / \"models\" / \"registry.json\"\n",
    "with open(registry_path) as f:\n",
    "    registry = json.load(f)\n",
    "\n",
    "if registry.get(\"latest\"):\n",
    "    latest = registry[\"latest\"]\n",
    "    model = LightGBMModel()\n",
    "    model.load(ROOT / \"models\" / \"artifacts\" / latest)\n",
    "    print(f\"Loaded model version: {latest}\")\n",
    "else:\n",
    "    print(\"No trained model found. Run trainer.py first.\")\n",
    "    print(\"Continuing with synthetic predictions for demonstration.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "e7309dd1",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load sample OHLCV data\n",
    "TICKERS = [\"RELIANCE\", \"TCS\", \"INFY\"]\n",
    "raw_dir = ROOT / \"storage\" / \"raw\"\n",
    "\n",
    "frames = {}\n",
    "for ticker in TICKERS:\n",
    "    csv_path = raw_dir / f\"{ticker}.csv\"\n",
    "    if csv_path.exists():\n",
    "        frames[ticker] = pd.read_csv(csv_path, parse_dates=[\"Date\"], index_col=\"Date\")\n",
    "        print(f\"Loaded {ticker}: {len(frames[ticker])} rows\")\n",
    "    else:\n",
    "        print(f\"Missing {csv_path} â€” generating synthetic data for demo\")\n",
    "        dates = pd.bdate_range(end=pd.Timestamp.today(), periods=252)\n",
    "        np.random.seed(hash(ticker) % 2**31)\n",
    "        close = 1000 + np.cumsum(np.random.randn(252) * 10)\n",
    "        frames[ticker] = pd.DataFrame({\n",
    "            \"Open\": close + np.random.randn(252),\n",
    "            \"High\": close + abs(np.random.randn(252) * 5),\n",
    "            \"Low\": close - abs(np.random.randn(252) * 5),\n",
    "            \"Close\": close,\n",
    "            \"Volume\": np.random.randint(100000, 5000000, 252),\n",
    "        }, index=dates)\n",
    "\n",
    "print(f\"\\nLoaded {len(frames)} tickers\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d9a941b8",
   "metadata": {},
   "source": [
    "## 2. Generate Predictions and Run Paper Simulator"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "dbeae1da",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Run paper simulation\n",
    "sim = PaperSimulator(initial_cash=1_000_000.0)\n",
    "\n",
    "# Generate synthetic signals for demonstration\n",
    "np.random.seed(42)\n",
    "trade_log = []\n",
    "\n",
    "for ticker, df in frames.items():\n",
    "    for i, (date, row) in enumerate(df.iterrows()):\n",
    "        if i < 20:  # skip warm-up period\n",
    "            continue\n",
    "        # Simulate a simple signal: random buy/sell/hold\n",
    "        signal = np.random.choice([\"buy\", \"sell\", \"hold\"], p=[0.15, 0.10, 0.75])\n",
    "        if signal == \"hold\":\n",
    "            continue\n",
    "        intent = OrderIntent(\n",
    "            ticker=ticker,\n",
    "            side=signal,\n",
    "            quantity=10,\n",
    "            price=row[\"Close\"],\n",
    "            timestamp=date,\n",
    "        )\n",
    "        fill = sim.execute_intent(intent)\n",
    "        if fill:\n",
    "            trade_log.append({\n",
    "                \"date\": date,\n",
    "                \"ticker\": ticker,\n",
    "                \"side\": signal,\n",
    "                \"price\": fill.fill_price,\n",
    "                \"quantity\": fill.quantity,\n",
    "                \"pnl\": getattr(fill, \"pnl\", 0.0),\n",
    "            })\n",
    "\n",
    "trades_df = pd.DataFrame(trade_log)\n",
    "print(f\"Total trades executed: {len(trades_df)}\")\n",
    "trades_df.head(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "57204817",
   "metadata": {},
   "source": [
    "## 3. Portfolio Equity Curve"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "4b9b8e42",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Build equity curve from audit log\n",
    "audit = sim.audit_log if hasattr(sim, \"audit_log\") else []\n",
    "\n",
    "if audit:\n",
    "    equity_df = pd.DataFrame(audit)\n",
    "    equity_df.set_index(\"timestamp\", inplace=True)\n",
    "else:\n",
    "    # Fallback: reconstruct from trade log\n",
    "    equity = [1_000_000.0]\n",
    "    dates = [frames[TICKERS[0]].index[20]]\n",
    "    cumulative = 1_000_000.0\n",
    "    for _, trade in trades_df.iterrows():\n",
    "        delta = trade[\"quantity\"] * trade[\"price\"] * (1 if trade[\"side\"] == \"sell\" else -1)\n",
    "        cumulative += delta * 0.001  # simplified P&L proxy\n",
    "        equity.append(cumulative)\n",
    "        dates.append(trade[\"date\"])\n",
    "    equity_df = pd.DataFrame({\"equity\": equity}, index=dates)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(14, 5))\n",
    "ax.plot(equity_df.index, equity_df.iloc[:, 0], linewidth=1.2, color=\"steelblue\")\n",
    "ax.set_title(\"Paper Trading â€” Portfolio Equity Curve\", fontsize=14)\n",
    "ax.set_xlabel(\"Date\")\n",
    "ax.set_ylabel(\"Portfolio Value (â‚¹)\")\n",
    "ax.grid(True, alpha=0.3)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

